In [ ]:
import requests
import pandas as pd
import time
import re

# =========================================================
# PART 1: Fetch 5000 English Steam reviews for God of War (2018)
# =========================================================

def fetch_steam_reviews(app_id, target_n=5000, language="english", sleep_sec=1.2):
    base_url = f"https://store.steampowered.com/appreviews/{app_id}"
    cursor = "*"
    collected = []
    seen_ids = set()
    page = 1

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    while len(collected) < target_n:
        params = {
            "json": 1,
            "filter": "updated",
            "language": language,
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor
        }

        try:
            response = session.get(base_url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"[ERROR] Page {page} failed: {e}")
            time.sleep(5)
            continue

        reviews = data.get("reviews", [])
        next_cursor = data.get("cursor")

        if not reviews:
            print(f"[STOP] No more reviews on page {page}.")
            break

        added = 0

        for r in reviews:
            review_id = r.get("recommendationid")
            if review_id in seen_ids:
                continue

            seen_ids.add(review_id)
            author = r.get("author", {}) or {}

            collected.append({
                "app_id": app_id,
                "recommendationid": review_id,
                "review_text": r.get("review", ""),
                "voted_up": r.get("voted_up"),
                "timestamp_created": r.get("timestamp_created"),
                "timestamp_updated": r.get("timestamp_updated"),
                "votes_up": r.get("votes_up"),
                "votes_funny": r.get("votes_funny"),
                "weighted_vote_score": r.get("weighted_vote_score"),
                "comment_count": r.get("comment_count"),
                "steam_purchase": r.get("steam_purchase"),
                "received_for_free": r.get("received_for_free"),
                "written_during_early_access": r.get("written_during_early_access"),
                "author_steamid": author.get("steamid"),
                "author_num_games_owned": author.get("num_games_owned"),
                "author_num_reviews": author.get("num_reviews"),
                "author_playtime_forever_min": author.get("playtime_forever"),
                "author_playtime_last_two_weeks_min": author.get("playtime_last_two_weeks"),
                "author_playtime_at_review_min": author.get("playtime_at_review"),
                "author_last_played": author.get("last_played")
            })

            added += 1
            if len(collected) >= target_n:
                break

        print(f"Page {page}: added {added}, total = {len(collected)}/{target_n}")

        if len(collected) >= target_n:
            break

        if not next_cursor or next_cursor == cursor:
            print("[STOP] Cursor did not advance.")
            break

        cursor = next_cursor
        page += 1
        time.sleep(sleep_sec)

    return pd.DataFrame(collected)


def clean_reviews_10h(df, min_hours=10, use_playtime_at_review=True):
    df = df.copy()

    df["review_text"] = df["review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["review_text"] != ""]

    playtime_col = "author_playtime_at_review_min" if use_playtime_at_review else "author_playtime_forever_min"
    df[playtime_col] = pd.to_numeric(df[playtime_col], errors="coerce")

    df["playtime_hours"] = df[playtime_col] / 60
    df = df[df["playtime_hours"] >= min_hours]

    df = df.drop_duplicates(subset=["recommendationid"]).reset_index(drop=True)
    return df


def build_20word_dataset(df_master):
    df_master = df_master.copy()
    df_master["review_text"] = df_master["review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df_master["word_count"] = df_master["review_text"].str.split().str.len()
    df_20 = df_master[df_master["word_count"] >= 20].copy().reset_index(drop=True)
    return df_20


# =========================================================
# PART 2: Steam narrative analysis for God of War (2018)
# =========================================================

def run_gow2018_narrative_analysis(input_csv):
    df = pd.read_csv(input_csv)
    text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"
    df[text_col] = df[text_col].astype(str).str.lower()

    # God of War (2018)-specific broad narrative dictionary
    narrative_dict = {
        "story": ["story", "narrative", "plot"],
        "characters": [
            "character", "characters",
            "kratos", "atreus", "mimir", "baldur", "freya", "brok", "sindri"
        ],
        "ending": ["ending", "finale"],
        "writing": ["writing", "dialogue"],
        "immersion": ["immersive", "immersion", "world", "journey"],
        "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
    }

    for category, words in narrative_dict.items():
        pattern = r"\b(?:"
        pattern += "|".join(re.escape(w) for w in words)
        pattern += r")\b"
        df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

    narrative_cols = list(narrative_dict.keys())
    df["narrative_broad"] = df[narrative_cols].any(axis=1)

    pos_df = df[df["voted_up"] == True].copy()
    neg_df = df[df["voted_up"] == False].copy()
    nar_df = df[df["narrative_broad"]].copy()

    print("Total reviews:", len(df))
    print("Positive reviews:", len(pos_df))
    print("Negative reviews:", len(neg_df))

    print("\n=== Q1: Narrative in overall positive reviews ===")
    print("Positive reviews mentioning any broad narrative element:",
          pos_df["narrative_broad"].sum())
    print("Share of positive reviews mentioning narrative:",
          round(pos_df["narrative_broad"].mean() * 100, 2), "%")

    print("\nNarrative category presence within positive reviews:")
    q1_rows = []
    for col in narrative_cols:
        count = int(pos_df[col].sum())
        pct = round(pos_df[col].mean() * 100, 2)
        q1_rows.append({
            "category": col,
            "positive_count": count,
            "positive_pct": pct
        })
        print(f"{col}: {count} ({pct}%)")

    q1_df = pd.DataFrame(q1_rows).sort_values("positive_count", ascending=False)

    print("\n=== Narrative-related reviews sentiment split ===")
    print(nar_df["voted_up"].value_counts(dropna=False))
    print("Positive share among narrative-related reviews:",
          round((nar_df["voted_up"] == True).mean() * 100, 2), "%")

    print("\n=== Q2: Broad narrative elements co-occurring with positive experience ===")
    print(q1_df)

    positive_words = [
        "amazing", "masterpiece", "love", "memorable", "immersive",
        "beautiful", "incredible", "best", "favorite", "great", "emotional"
    ]

    for w in positive_words:
        pattern = r"\b" + re.escape(w) + r"\b"
        df[w] = df[text_col].str.contains(pattern, regex=True, na=False)

    pos_df = df[df["voted_up"] == True].copy()

    co_results = []
    for nar in narrative_cols:
        for pw in positive_words:
            count = int(((pos_df[nar]) & (pos_df[pw])).sum())
            if count > 0:
                co_results.append({
                    "narrative_element": nar,
                    "positive_word": pw,
                    "count": count
                })

    co_df = pd.DataFrame(co_results).sort_values("count", ascending=False)

    print("\nTop co-occurrences between narrative elements and positive experience words:")
    print(co_df.head(30))

    neg_rows = []
    for col in narrative_cols:
        count = int(neg_df[col].sum())
        pct = round(neg_df[col].mean() * 100, 2)
        neg_rows.append({
            "category": col,
            "negative_count": count,
            "negative_pct": pct
        })

    neg_q_df = pd.DataFrame(neg_rows).sort_values("negative_count", ascending=False)
    print("\nNarrative category presence within negative reviews:")
    print(neg_q_df)

    q1_df.to_csv("gow2018_q1_positive_narrative_presence.csv", index=False, encoding="utf-8-sig")
    co_df.to_csv("gow2018_q2_narrative_positive_cooccurrence.csv", index=False, encoding="utf-8-sig")
    neg_q_df.to_csv("gow2018_negative_narrative_presence.csv", index=False, encoding="utf-8-sig")

    return df, q1_df, co_df, neg_q_df


# =========================================================
# PART 3: Run all steps
# =========================================================

if __name__ == "__main__":
    APP_ID = 1593500  # God of War (2018) on Steam

    # Step 1: fetch raw 5000 English reviews
    df_raw = fetch_steam_reviews(
        app_id=APP_ID,
        target_n=5000,
        language="english",
        sleep_sec=1.2
    )
    df_raw.to_csv("gow2018_reviews_raw_english_5000.csv", index=False, encoding="utf-8-sig")

    # Step 2: clean to 10h+
    df_master = clean_reviews_10h(df_raw, min_hours=10, use_playtime_at_review=True)
    print("\nMaster rows:", len(df_master))
    df_master.to_csv("gow2018_reviews_clean_english_5000_10h.csv", index=False, encoding="utf-8-sig")

    # Step 3: build 20-word analysis dataset
    df_20 = build_20word_dataset(df_master)
    print("Rebuilt >=20 words rows:", len(df_20))
    df_20.to_csv("gow2018_reviews_clean_english_5000_10h_20words_REBUILT.csv", index=False, encoding="utf-8-sig")

    # Step 4: run Steam narrative analysis
    run_gow2018_narrative_analysis("gow2018_reviews_clean_english_5000_10h_20words_REBUILT.csv")

In [ ]:
import pandas as pd
import re

df = pd.read_csv("gow2018_reviews_clean_english_5000_10h_20words_REBUILT.csv")
text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"
df[text_col] = df[text_col].astype(str).str.lower()

narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": [
        "character", "characters",
        "kratos", "atreus", "mimir", "baldur", "freya", "brok", "sindri"
    ],
    "ending": ["ending", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

for category, words in narrative_dict.items():
    pattern = r"\b(?:"
    pattern += "|".join(re.escape(w) for w in words)
    pattern += r")\b"
    df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

narrative_cols = list(narrative_dict.keys())
df["narrative_broad"] = df[narrative_cols].any(axis=1)

pos_df = df[df["voted_up"] == True].copy()
neg_df = df[df["voted_up"] == False].copy()
nar_df = df[df["narrative_broad"]].copy()

q1_rows = []
for col in narrative_cols:
    q1_rows.append({
        "category": col,
        "positive_count": int(pos_df[col].sum()),
        "positive_pct": round(pos_df[col].mean() * 100, 2)
    })

q1_df = pd.DataFrame(q1_rows).sort_values("positive_count", ascending=False)

positive_words = [
    "amazing", "masterpiece", "love", "memorable", "immersive",
    "beautiful", "incredible", "best", "favorite", "great", "emotional"
]

for w in positive_words:
    pattern = r"\b" + re.escape(w) + r"\b"
    df[w] = df[text_col].str.contains(pattern, regex=True, na=False)

pos_df = df[df["voted_up"] == True].copy()

co_results = []
for nar in narrative_cols:
    for pw in positive_words:
        count = int(((pos_df[nar]) & (pos_df[pw])).sum())
        if count > 0:
            co_results.append({
                "narrative_element": nar,
                "positive_word": pw,
                "count": count
            })

co_df = pd.DataFrame(co_results).sort_values("count", ascending=False)

neg_rows = []
for col in narrative_cols:
    neg_rows.append({
        "category": col,
        "negative_count": int(neg_df[col].sum()),
        "negative_pct": round(neg_df[col].mean() * 100, 2)
    })

neg_q_df = pd.DataFrame(neg_rows).sort_values("negative_count", ascending=False)

print("Total reviews:", len(df))
print("Positive reviews:", len(pos_df))
print("Negative reviews:", len(neg_df))

print("\n=== Q1: Narrative in overall positive reviews ===")
print("Positive reviews mentioning any broad narrative element:", pos_df["narrative_broad"].sum())
print("Share of positive reviews mentioning narrative:", round(pos_df["narrative_broad"].mean() * 100, 2), "%")

print("\nNarrative category presence within positive reviews:")
print(q1_df)

print("\n=== Narrative-related reviews sentiment split ===")
print(nar_df["voted_up"].value_counts(dropna=False))
print("Positive share among narrative-related reviews:", round((nar_df["voted_up"] == True).mean() * 100, 2), "%")

print("\nTop co-occurrences between narrative elements and positive experience words:")
print(co_df.head(20).to_string(index=False))

print("\nNarrative category presence within negative reviews:")
print(neg_q_df)

In [ ]:
print(co_df.head(20).to_string(index=False))

In [ ]:
import pandas as pd
import re

# ========= 1. 读取数据 =========
df = pd.read_csv("gow2018_reviews_clean_english_5000_10h_20words_REBUILT.csv")
text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"
df[text_col] = df[text_col].astype(str).str.lower()

# ========= 2. 定义叙事词表 =========
narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": [
        "character", "characters",
        "kratos", "atreus", "mimir", "baldur", "freya", "brok", "sindri"
    ],
    "ending": ["ending", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

for category, words in narrative_dict.items():
    pattern = r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b"
    df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

narrative_cols = list(narrative_dict.keys())
df["narrative_broad"] = df[narrative_cols].any(axis=1)

# ========= 3. 定义机制缺陷词 =========
gow_flaw_terms = [
    "combat", "gameplay", "controls", "control", "clunky",
    "repetitive", "boring", "tedious", "slow", "outdated",
    "floaty", "janky", "grindy", "difficulty", "leveling",
    "loot", "inventory", "riding", "backtracking", "linear"
]

flaw_pattern = r"\b(?:" + "|".join(re.escape(x) for x in gow_flaw_terms) + r")\b"
df["mechanical_flaw"] = df[text_col].str.contains(flaw_pattern, regex=True, na=False)

# ========= 4. 分组 =========
flaw_df = df[df["mechanical_flaw"]].copy()
flaw_pos = flaw_df[flaw_df["voted_up"] == True].copy()
flaw_neg = flaw_df[flaw_df["voted_up"] == False].copy()

# ========= 5. 总体结果 =========
print("Total reviews:", len(df))
print("Mechanical flaw comments:", len(flaw_df))

print("\n=== Mechanical flaw comments ===")
print("Positive flaw comments:", len(flaw_pos))
print("Negative flaw comments:", len(flaw_neg))

print("\nNarrative broad share in positive flaw comments:")
print(round(flaw_pos["narrative_broad"].mean() * 100, 2), "%")

print("Narrative broad share in negative flaw comments:")
print(round(flaw_neg["narrative_broad"].mean() * 100, 2), "%")

# ========= 6. flaw comments里的narrative categories =========
rows = []
for col in narrative_cols:
    rows.append({
        "category": col,
        "positive_flaw_count": int(flaw_pos[col].sum()),
        "positive_flaw_pct": round(flaw_pos[col].mean() * 100, 2) if len(flaw_pos) > 0 else 0,
        "negative_flaw_count": int(flaw_neg[col].sum()),
        "negative_flaw_pct": round(flaw_neg[col].mean() * 100, 2) if len(flaw_neg) > 0 else 0
    })

comp_df = pd.DataFrame(rows).sort_values("positive_flaw_pct", ascending=False)

print("\n=== Narrative categories in flaw comments ===")
print(comp_df.to_string(index=False))

# ========= 7. 导出表 =========
comp_df.to_csv("gow2018_narrative_compensation_flaw_comparison.csv", index=False, encoding="utf-8-sig")
flaw_pos.to_csv("gow2018_positive_flaw_comments.csv", index=False, encoding="utf-8-sig")
flaw_neg.to_csv("gow2018_negative_flaw_comments.csv", index=False, encoding="utf-8-sig")

# ========= 8. 看“but”句式 =========
but_pos = flaw_pos[flaw_pos[text_col].str.contains(r"\bbut\b", regex=True, na=False)].copy()
but_neg = flaw_neg[flaw_neg[text_col].str.contains(r"\bbut\b", regex=True, na=False)].copy()

print("\nPositive flaw comments with 'but':", len(but_pos))
print("Negative flaw comments with 'but':", len(but_neg))

print("\n=== Sample positive flaw comments with 'but' ===")
if len(but_pos) > 0:
    print(but_pos[[text_col]].head(10).to_string(index=False))
else:
    print("None")

print("\n=== Sample negative flaw comments with 'but' ===")
if len(but_neg) > 0:
    print(but_neg[[text_col]].head(10).to_string(index=False))
else:
    print("None")

In [ ]:
neg_flaw_only = flaw_neg.copy()

neg_flaw_only.to_csv("gow2018_negative_flaw_comments.csv", index=False, encoding="utf-8-sig")

print("Negative flaw comments:", len(neg_flaw_only))
print(neg_flaw_only[["review_text"]].head(20).to_string(index=False))